# Abdomen CT — YOLO-Seg 2D Segmentasyon Pipeline
### Kaggle Notebook (Birincil) · Google Colab (İkincil)

Model ailesi seçimi: **YOLO26-seg · YOLO11-seg · YOLOv9-seg · YOLOv8-seg**
(tümü `ultralytics` paketiyle aynı API üzerinden çalışır — sadece ağırlık dosyası değişir).

> **GT uyarısı:** Poligon etiketleri gerçek organ/lezyon konturu DEĞİLDİR —
> bounding box'ın 4 köşesinden (dikdörtgen) üretilmiştir
> (bkz. `Faz3a_VeriHazirlik_YOLOSeg.ipynb`). Bu nedenle mask mAP/IoU metrikleri
> piksel-doğru segmentasyon kalitesini değil, **kutu-seviyeli lokalizasyonu**
> yansıtır.

**Kaggle'da çalıştırma:**
1. Sağ panelden Dataset ekle (yolo_seg klasörünü içeren dataset)
2. `Settings → Accelerator → GPU (P100/T4)` seç
3. Hücreleri sırayla çalıştır

**Colab'da çalıştırma:**
1. `Runtime → Change runtime type → GPU`
2. Kaggle Secrets'a `KAGGLE_USERNAME` + `KAGGLE_KEY` ekle

---

| Adım | Süre |
|------|------|
| Kurulum + Konfigürasyon | ~3 dk |
| Eğitim (100 epoch, T4) | ~3–5 saat |
| Inference + F1 | ~15 dk |

> **Not:** Eğitim session kesilirse `YOLO_CONTINUE = True` yapıp devam edin.

---
## 0. Ortam Tespiti

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f'Ortam : {env_name}')

---
## 1. Ortam Kurulumu (Colab için)

In [ ]:
if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print('Kaggle kimlik bilgileri Colab Secrets\'tan yüklendi')
    except Exception:
        from google.colab import files
        import json as _j, shutil as _sh
        print('kaggle.json dosyasını seçin:')
        _up = files.upload()
        _kp = Path.home() / '.kaggle' / 'kaggle.json'
        _kp.parent.mkdir(parents=True, exist_ok=True)
        for fname in _up:
            _sh.move(fname, str(_kp))
        os.chmod(str(_kp), 0o600)
        _kd = _j.loads(_kp.read_text())
        os.environ['KAGGLE_USERNAME'] = _kd['username']
        os.environ['KAGGLE_KEY']      = _kd['key']
        print('kaggle.json yüklendi')

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive bağlandı')
else:
    print('Kaggle/Local ortamı — Colab kurulumu atlandı')

---
## 2. Kütüphane Kurulumu

In [ ]:
print("Kütüphaneler kuruluyor...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'ultralytics', 'pillow', 'pandas', 'numpy', 'tqdm'],
    check=True
)
import importlib; importlib.invalidate_caches()

import ultralytics
print(f"ultralytics : {ultralytics.__version__}")

import torch
print(f"PyTorch     : {torch.__version__}")

---
## 3. Model Ailesi Seçimi

| Aile | Geçerli boyutlar | Ağırlık örneği |
|------|------------------|-----------------|
| `yolo26`  | n, s, m, l, x | `yolo26m-seg.pt`  |
| `yolo11`  | n, s, m, l, x | `yolo11m-seg.pt`  |
| `yolov9`  | c, e          | `yolov9c-seg.pt`  |
| `yolov8`  | n, s, m, l, x | `yolov8m-seg.pt`  |

In [ ]:
MODEL_FAMILY = 'yolo11'   # 'yolo26' | 'yolo11' | 'yolov9' | 'yolov8'
MODEL_SIZE   = 'm'        # aşağıdaki tabloya göre

SEG_MODEL_SIZES = {
    'yolo26': ['n', 's', 'm', 'l', 'x'],
    'yolo11': ['n', 's', 'm', 'l', 'x'],
    'yolov9': ['c', 'e'],
    'yolov8': ['n', 's', 'm', 'l', 'x'],
}

def resolve_seg_weights(family: str, size: str) -> str:
    if family not in SEG_MODEL_SIZES:
        raise ValueError(f"Bilinmeyen model ailesi: {family} — seçenekler: {list(SEG_MODEL_SIZES)}")
    if size not in SEG_MODEL_SIZES[family]:
        raise ValueError(f"{family}-seg için geçersiz boyut: {size} — seçenekler: {SEG_MODEL_SIZES[family]}")
    return f'{family}{size}-seg.pt'

SEG_WEIGHTS = resolve_seg_weights(MODEL_FAMILY, MODEL_SIZE)
print(f'Seçilen model ailesi : {MODEL_FAMILY}-seg  (boyut={MODEL_SIZE})')
print(f'Ağırlık dosyası      : {SEG_WEIGHTS}')

---
## 4. Konfigürasyon

**Kaggle**: `KAGGLE_DATASET_SLUG` = dataset klasör adı (`/kaggle/input/` altında, `seg_data/yolo_seg` içermeli).
**Colab**: Dataset Kaggle'dan indirilir.

In [ ]:
import os, sys, json, shutil, time
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import List

# ─── Kullanıcı Ayarları ────────────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen-acikveri'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen-acikveri'
GITHUB_URL          = 'https://github.com/ramazan2020/abdomen1.git'
FOLD = 0

@dataclass
class SegConfig:
    model: str = SEG_WEIGHTS
    img_size: int = 512
    batch_size: int = 16
    epochs: int = 100
    patience: int = 30
    lr0: float = 0.001
    lrf: float = 0.01
    weight_decay: float = 0.0005
    mosaic: float = 0.0
    mixup: float = 0.0
    fliplr: float = 0.0
    flipud: float = 0.0
    hsv_h: float = 0.0
    hsv_s: float = 0.0
    hsv_v: float = 0.4
    degrees: float = 10.0
    erasing: float = 0.1

SEG_CFG = SegConfig()
# ──────────────────────────────────────────────────────────────────────────

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

if IS_KAGGLE:
    DATA_DIR      = Path('/kaggle/input') / KAGGLE_DATASET_SLUG
    WORK_DIR      = Path('/kaggle/working')
    DRIVE_BASE    = None

elif IS_COLAB:
    DATA_DIR      = Path('/content/kaggle_data')
    WORK_DIR      = Path('/content')
    DRIVE_BASE    = Path('/content/drive/MyDrive/Abdomen')
    if DRIVE_BASE:
        DRIVE_BASE.mkdir(parents=True, exist_ok=True)

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    _proje   = Path(os.environ.get('TR_ABDOMEN_PROJE',  r'D:/makale-pdf/Proje'))
    DATA_DIR = Path(os.environ.get('TR_ABDOMEN_BASE',   str(_proje / 'abdomen')))
    WORK_DIR = Path(os.environ.get('ABDOMEN_OUT_DIR',   str(_proje / 'outputs')))
    DRIVE_BASE = None

SEG_RUNS_DIR  = WORK_DIR / 'runs' / 'seg'
DATASET_YAML  = WORK_DIR / 'dataset_seg.yaml'
OUTPUT_DIR    = WORK_DIR / 'output'

SEG_YOLO_DATA_DIR = Path(os.environ.get('ABDOMEN_SEG_YOLO_DIR', str(WORK_DIR / 'seg_data' / 'yolo_seg'))) if IS_LOCAL else DATA_DIR / 'seg_data' / 'yolo_seg'
SEG_FOLD_DIR   = SEG_YOLO_DATA_DIR / f'fold{FOLD}'
SEG_TRAIN_IMG  = SEG_FOLD_DIR / 'images' / 'train'
SEG_VAL_IMG    = SEG_FOLD_DIR / 'images' / 'val'
SEG_VAL_LBL    = SEG_FOLD_DIR / 'labels' / 'val'

SEG_RUNS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Ortam          : {env_name}')
print(f'DATA_DIR       : {DATA_DIR}  (var={DATA_DIR.exists()})')
print(f'WORK_DIR       : {WORK_DIR}  (var={WORK_DIR.exists()})')
print(f'SEG_YOLO_DATA_DIR : {SEG_YOLO_DATA_DIR}  (var={SEG_YOLO_DATA_DIR.exists()})')
print(f'SEG_FOLD_DIR   : {SEG_FOLD_DIR}  (var={SEG_FOLD_DIR.exists()})')
print(f'Model          : {SEG_CFG.model}')

if not DATA_DIR.exists():
    raise FileNotFoundError(f'DATA_DIR bulunamadı: {DATA_DIR}')

In [ ]:
if IS_LOCAL:
    REPO_DIR = Path('.').resolve()
    print(f'Local: src/ kullanılıyor → {REPO_DIR / "src"}')
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        print(f'Klonlanıyor: {GITHUB_URL}')
        subprocess.run(
            ['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)],
            check=True
        )
    else:
        print('Repo mevcut, güncelleniyor...')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.evaluation import f1_at_iou, top5_f1_mean

print('src.evaluation  ✓  (f1_at_iou, top5_f1_mean)')

print(f'Repo : {REPO_DIR}')
print(f'sys.path[0] = {sys.path[0]}')

---
## 5. Veri Kontrolü

`seg_data/yolo_seg` **Faz3a_VeriHazirlik_YOLOSeg.ipynb** tarafından oluşturulur.
Bu hücre yalnızca doğrulama yapar.

In [ ]:
if not SEG_FOLD_DIR.exists():
    raise FileNotFoundError(
        f'YOLO-seg verisi bulunamadı: {SEG_FOLD_DIR}\n'
        'Önce Faz3a_VeriHazirlik_YOLOSeg.ipynb dosyasını çalıştırın.'
    )

n_train = len(list(SEG_TRAIN_IMG.glob('*.png'))) if SEG_TRAIN_IMG.exists() else 0
n_val   = len(list(SEG_VAL_IMG.glob('*.png')))   if SEG_VAL_IMG.exists()   else 0

if n_train == 0:
    raise FileNotFoundError(
        f'PNG bulunamadı: {SEG_TRAIN_IMG}\n'
        'Önce Faz3a_VeriHazirlik_YOLOSeg.ipynb dosyasını çalıştırın.'
    )

print(f'yolo_seg verisi mevcut ✓')
print(f'  train PNG : {n_train:,}')
print(f'  val   PNG : {n_val:,}')

---
## 6. Veri İndirme (Yalnızca Colab)

**Kaggle'da bu hücre otomatik atlanır** — PNG'ler dataset içinde hazır olmalı.

In [ ]:
if IS_COLAB:
    _already = SEG_FOLD_DIR.exists() and SEG_TRAIN_IMG.exists()
    if _already:
        print(f"Veri zaten mevcut: {SEG_FOLD_DIR}")
    else:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Dataset indiriliyor: {KAGGLE_DATASET_ID}")
        print("Bu işlem 30-90 dakika sürebilir...")
        t0 = time.time()
        r = subprocess.run(
            ['kaggle', 'datasets', 'download',
             '-d', KAGGLE_DATASET_ID,
             '-p', str(DATA_DIR), '--unzip'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('HATA:', r.stderr[-1000:])
            raise RuntimeError('İndirme başarısız')
        print(f"İndirildi ({time.time()-t0:.0f}s)")

assert SEG_FOLD_DIR.exists(), f'YOLO-seg fold dizini bulunamadı: {SEG_FOLD_DIR}'

n_train = len(list(SEG_TRAIN_IMG.glob('*.png'))) if SEG_TRAIN_IMG.exists() else 0
n_val   = len(list(SEG_VAL_IMG.glob('*.png')))   if SEG_VAL_IMG.exists()   else 0
print(f'train PNG : {n_train:,}')
print(f'val   PNG : {n_val:,}')

if n_train == 0:
    raise FileNotFoundError(
        f'PNG bulunamadı: {SEG_TRAIN_IMG}\n'
        'Dataset içinde seg_data/yolo_seg/ klasörü eksik olabilir.'
    )

---
## 7. dataset.yaml Doğrula

`Faz3a_VeriHazirlik_YOLOSeg.ipynb` zaten `SEG_FOLD_DIR/dataset.yaml` yazmıştır; doğrudan onu kullanırız.

In [ ]:
DATASET_YAML = SEG_FOLD_DIR / 'dataset.yaml'
if not DATASET_YAML.exists():
    raise FileNotFoundError(f'dataset.yaml bulunamadı: {DATASET_YAML}')
print(f'dataset.yaml: {DATASET_YAML}')
print(DATASET_YAML.read_text())

---
## 8. Eğitim

`-seg` ağırlığı kullanıldığında ultralytics görevi otomatik olarak `segment` algılar.
Checkpoint otomatik `runs/seg/` altına kaydedilir.

In [ ]:
from ultralytics import YOLO
import torch

SEG_CONTINUE = False  # True → son checkpoint'ten devam

_run_name = f'fold{FOLD}_{Path(SEG_CFG.model).stem}'
print(f'Eğitim : {_run_name}')
print(f'Runs   : {SEG_RUNS_DIR}')

if torch.backends.mps.is_available():
    _device, _workers = 'mps', 2
elif torch.cuda.is_available():
    _device, _workers = '0', 8
else:
    _device, _workers = 'cpu', 4

seg_model = YOLO(SEG_CFG.model)
seg_model.train(
    data=str(DATASET_YAML),
    imgsz=SEG_CFG.img_size,
    epochs=SEG_CFG.epochs,
    batch=SEG_CFG.batch_size,
    patience=SEG_CFG.patience,
    lr0=SEG_CFG.lr0,
    lrf=SEG_CFG.lrf,
    weight_decay=SEG_CFG.weight_decay,
    mosaic=SEG_CFG.mosaic,
    mixup=SEG_CFG.mixup,
    fliplr=SEG_CFG.fliplr,
    flipud=SEG_CFG.flipud,
    hsv_h=SEG_CFG.hsv_h,
    hsv_s=SEG_CFG.hsv_s,
    hsv_v=SEG_CFG.hsv_v,
    degrees=SEG_CFG.degrees,
    erasing=SEG_CFG.erasing,
    project=str(SEG_RUNS_DIR),
    name=_run_name,
    device=_device,
    workers=_workers,
    seed=42,
    deterministic=False,
    resume=SEG_CONTINUE,
    amp=True,
)

SEG_BEST = Path(seg_model.trainer.save_dir) / 'weights' / 'best.pt'
print(f'\nEn iyi ağırlık: {SEG_BEST}')

if IS_COLAB and DRIVE_BASE:
    _yd = DRIVE_BASE / 'seg_weights'
    _yd.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(SEG_BEST), str(_yd / f'fold{FOLD}_best.pt'))
    print(f"Drive'a yedeklendi: {_yd}")

---
## 8b. Per-Class mAP (Box + Mask — YOLO Native Val)

> Mask mAP burada GT dikdörtgen poligonuna karşı ölçülür — gerçek organ konturu
> temsil ETMEZ; sadece kutu-seviyeli lokalizasyonun maske formunda ölçümüdür.

In [ ]:
from ultralytics import YOLO
import pandas as pd

_best = Path(seg_model.trainer.save_dir) / 'weights' / 'best.pt' if hasattr(seg_model, 'trainer') else SEG_BEST
val_model = YOLO(str(_best))

val_results = val_model.val(data=str(DATASET_YAML), split='val', verbose=False)

_names = val_results.names
_box_ap50 = val_results.box.ap50
_box_maps = val_results.box.maps
_msk_ap50 = val_results.seg.ap50
_msk_maps = val_results.seg.maps

rows = []
for idx, name in _names.items():
    rows.append({
        'Class'            : name,
        'Box mAP@0.5'      : round(float(_box_ap50[idx]), 4),
        'Box mAP@0.5:0.95' : round(float(_box_maps[idx]), 4),
        'Mask mAP@0.5'     : round(float(_msk_ap50[idx]), 4),
        'Mask mAP@0.5:0.95': round(float(_msk_maps[idx]), 4),
    })
df_map = pd.DataFrame(rows)
df_map.loc[len(df_map)] = {
    'Class': 'ALL (mean)',
    'Box mAP@0.5': round(float(val_results.box.map50), 4),
    'Box mAP@0.5:0.95': round(float(val_results.box.map), 4),
    'Mask mAP@0.5': round(float(val_results.seg.map50), 4),
    'Mask mAP@0.5:0.95': round(float(val_results.seg.map), 4),
}

print(f'=== YOLO-Seg Per-Class mAP — Fold {FOLD} ===')
print(df_map.to_string(index=False))

df_map.to_csv(OUTPUT_DIR / f'yoloseg_fold{FOLD}_map.csv', index=False)
print(f'\nKaydedildi: {OUTPUT_DIR}/yoloseg_fold{FOLD}_map.csv')

---
## 9. Tahmin (Inference) — Validasyon Seti

In [ ]:
from ultralytics import YOLO
from PIL import Image
from tqdm.notebook import tqdm

assert 'SEG_BEST' in dir() and SEG_BEST.exists(), f'Model bulunamadı: {SEG_BEST}'

def _parse_stem(stem: str):
    parts = stem.split('_')
    if len(parts) >= 3:
        return int(parts[1]), int(parts[2])
    return int(parts[0]), int(parts[1])

_model = YOLO(str(SEG_BEST))
_val_imgs = sorted(SEG_VAL_IMG.glob('*.png'))
print(f'Val inference: {len(_val_imgs)} PNG')

CONF_TH = 0.05
seg_preds = []
for ip in tqdm(_val_imgs, desc='YOLO-Seg inference'):
    case, image_id = _parse_stem(ip.stem)
    res = _model.predict(str(ip), conf=CONF_TH, verbose=False,
                          imgsz=SEG_CFG.img_size)[0]
    for box, sc, cl in zip(res.boxes.xyxy.cpu().numpy(),
                            res.boxes.conf.cpu().numpy(),
                            res.boxes.cls.cpu().numpy()):
        seg_preds.append({
            'case': case, 'image_id': image_id, 'class': int(cl),
            'score': float(sc),
            'x1': float(box[0]), 'y1': float(box[1]),
            'x2': float(box[2]), 'y2': float(box[3]),
        })

seg_pred_df = pd.DataFrame(seg_preds)
print(f'Tahmin: {len(seg_pred_df):,} kutu  |  {seg_pred_df["case"].nunique() if not seg_pred_df.empty else 0} vaka')

---
## 10. Ground Truth Yükleme (Poligon → Kutu)

In [ ]:
gt_rows = []
for lp in sorted(SEG_VAL_LBL.glob('*.txt')):
    ip = SEG_VAL_IMG / (lp.stem + '.png')
    if not ip.exists():
        continue
    W, H = Image.open(ip).size
    case, image_id = _parse_stem(lp.stem)
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 9:
            continue
        cl = int(p[0])
        coords = list(map(float, p[1:9]))
        xs = [coords[i]*W for i in range(0, 8, 2)]
        ys = [coords[i]*H for i in range(1, 8, 2)]
        gt_rows.append({
            'case': case, 'image_id': image_id, 'class': cl,
            'x1': min(xs), 'y1': min(ys), 'x2': max(xs), 'y2': max(ys),
        })

gt_df = pd.DataFrame(gt_rows)
print(f'GT: {len(gt_df):,} kutu (poligon→bbox)  |  {gt_df["case"].nunique() if not gt_df.empty else 0} vaka')
if not gt_df.empty:
    print(gt_df.groupby('class').size()
            .rename(index={i: SUPER_CLASSES[i] for i in range(len(SUPER_CLASSES))}))

---
## 11. Değerlendirme — F1 Skoru (Box-IoU, Faz3b ile karşılaştırılabilir)

In [ ]:
if seg_pred_df.empty:
    print('Prediction boş — inference adımını kontrol edin.')
else:
    top5   = top5_f1_mean(seg_pred_df, gt_df)
    detail = f1_at_iou(seg_pred_df, gt_df, iou_th=0.3)

    print(f'=== YOLO-Seg — Fold {FOLD} Sonuçları ===')
    print(f'Top-5 Mean F1      : {top5["top5_mean_f1"]:.4f}')
    print(f'Macro F1 @ IoU=0.3 : {detail["macro_f1"]:.4f}')
    print(f'Micro F1 @ IoU=0.3 : {detail["micro_f1"]:.4f}')
    print()
    for cls_id, cls_name in enumerate(SUPER_CLASSES):
        if cls_id in detail['per_class']:
            m = detail['per_class'][cls_id]
            print(f'  {cls_id} {cls_name:<30}  '
                  f'P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}  '
                  f'TP={m["tp"]} FP={m["fp"]} FN={m["fn"]}')

---
## 12. Görsel Kontrol (Maske + GT Poligon)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
%matplotlib inline

N_SAMPLES  = 4
CONF_SHOW  = 0.25

random.seed(42)
_val_imgs_all = sorted(SEG_VAL_IMG.glob('*.png'))
_annotated = [ip for ip in _val_imgs_all
              if (SEG_VAL_LBL / (ip.stem + '.txt')).exists()
              and (SEG_VAL_LBL / (ip.stem + '.txt')).stat().st_size > 0]
_samples = random.sample(_annotated, min(N_SAMPLES, len(_annotated)))

fig, axes = plt.subplots(1, len(_samples), figsize=(5*len(_samples), 5))
if len(_samples) == 1: axes = [axes]

for ax, ip in zip(axes, _samples):
    img = np.array(Image.open(ip).convert('RGB'))
    ax.imshow(img)
    case, image_id = _parse_stem(ip.stem)

    for _, r in gt_df[(gt_df['case']==case) & (gt_df['image_id']==image_id)].iterrows():
        ax.add_patch(mpatches.Rectangle(
            (r.x1, r.y1), r.x2-r.x1, r.y2-r.y1,
            linewidth=2, edgecolor='cyan', facecolor='none', linestyle='--'
        ))
        ax.text(r.x1, r.y1-4, SUPER_CLASSES[int(r['class'])][:12],
                color='cyan', fontsize=7, backgroundcolor='black')

    _res = _model.predict(str(ip), conf=CONF_SHOW, verbose=False, imgsz=SEG_CFG.img_size)[0]
    if _res.masks is not None:
        for mask_xy, cl in zip(_res.masks.xy, _res.boxes.cls.cpu().numpy()):
            ax.add_patch(mpatches.Polygon(mask_xy, closed=True,
                                           edgecolor='red', facecolor='red', alpha=0.3, linewidth=2))

    ax.set_title(f'case={case} slice={image_id}', fontsize=8)
    ax.axis('off')

plt.suptitle(f'Cyan=GT poligon(kesikli)  Red=Pred mask(conf≥{CONF_SHOW})  Fold {FOLD}', fontsize=11)
plt.tight_layout()
plt.show()

---
## 13. Sonuçları Kaydet

In [ ]:
if seg_pred_df.empty:
    print('Prediction boş, kaydetme atlandı.')
else:
    seg_pred_df.to_csv(OUTPUT_DIR / f'yoloseg_fold{FOLD}_pred.csv', index=False)

    summary = {
        'fold'          : FOLD,
        'model'         : SEG_CFG.model,
        'top5_mean_f1'  : top5['top5_mean_f1'],
        'macro_f1_03'   : detail['macro_f1'],
        'micro_f1_03'   : detail['micro_f1'],
        'per_class_03'  : {
            SUPER_CLASSES[cid]: {
                k: round(v, 4) if isinstance(v, float) else v
                for k, v in m.items()
            }
            for cid, m in detail['per_class'].items()
        },
    }
    (OUTPUT_DIR / f'yoloseg_fold{FOLD}_summary.json').write_text(
        json.dumps(summary, indent=2, ensure_ascii=False)
    )

    print(f'Çıktılar: {OUTPUT_DIR}')
    for f in sorted(OUTPUT_DIR.glob('yoloseg_*')):
        print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

    if IS_COLAB and DRIVE_BASE:
        _dst = DRIVE_BASE / 'output'
        _dst.mkdir(parents=True, exist_ok=True)
        for f in OUTPUT_DIR.glob('yoloseg_*'):
            shutil.copy2(str(f), str(_dst / f.name))
        print(f"Drive'a kopyalandı: {_dst}")

    if IS_KAGGLE:
        print('\nKaggle çıktıları kaydetmek için:')
        print('  Save Version → sağ üst → Save & Run All')